# 04 — Fraud injection and detection power

Phase 5 ties Phases 1–4 together: take a clean Benford-conforming dataset (world
city populations), inject fabricated values at increasing rates, and watch the
four-test conformity bundle cross from *accept* to *reject*. The publication
figures live in `figures/fraud_before_after.png` and
`figures/fraud_detection_power.png` (generated by `scripts/exp_fraud_demo.py`).

This notebook is the exploratory companion. Three sanity checks:

1. **Three fabrication strategies, side-by-side.** Compare the first-digit
   profile of `uniform_digit`, `round_numbers`, and `psychological` synthesis.
2. **Single before/after panel** at 30 % `uniform_digit` contamination.
3. **Detection-power sweep** at modest `n_trials` so the cell stays fast (the
   article-quality version uses `n_trials = 30` and ~21 fractions; this one is
   tuned for interactive feedback).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.benford import DIGITS, benford_pmf, empirical_frequencies
from src.conformity import conformity_report
from src.datasets import load_world_cities
from src.fraud import detection_power_curve, fabricate_values, inject_fraud

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = REPO_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

PMF = benford_pmf()

## 1. Three fabrication strategies

Each fabrication kind violates Benford in a *different* way:

- **uniform_digit:** every leading digit appears ~11.1 %.
- **round_numbers:** values cluster on $\{100, 200, 250, 500, 1000, ...\}$,
  inflating $d \in \{1, 2, 5\}$.
- **psychological:** humans over-pick mid digits 3–6 and under-pick 1 and 9.

In [ ]:
rng = np.random.default_rng(0)
kinds = ["uniform_digit", "round_numbers", "psychological"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, kind in zip(axes, kinds):
    fake = fabricate_values(20_000, kind=kind, magnitude_window=(1e2, 1e6), rng=rng)
    freq = empirical_frequencies(fake)
    ax.bar(DIGITS - 0.18, freq, width=0.36, label="Empirical", color="#a83b3b")
    ax.bar(DIGITS + 0.18, PMF, width=0.36, label="Benford", color="#d68a3e")
    ax.set_xticks(DIGITS)
    ax.set_xlabel("First digit $d$")
    ax.set_title(kind)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
axes[0].set_ylabel("Probability")
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Three fabrication strategies, $n = 20{,}000$", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## 2. Before vs after at 30 % contamination

Take the clean GeoNames city-populations dataset, inject 30 % `uniform_digit`
fabricated values, and run the four-test bundle on each side. The cooked
dataset must be flagged.

In [ ]:
try:
    cities = load_world_cities(min_population=1)
    pop = cities["population"].to_numpy(dtype=float)
    contaminated = inject_fraud(pop, fraction=0.3, kind="uniform_digit", rng=np.random.default_rng(0))

    print("=== Clean ===")
    print(conformity_report(pop).summary_table())
    print()
    print("=== Contaminated (30% uniform_digit) ===")
    print(conformity_report(contaminated).summary_table())
except FileNotFoundError as exc:
    print(f"SKIPPED: {exc}")
    print("Run `python scripts/build_datasets.py` from the repo root and re-run this cell.")

## 3. Detection-power sweep (fast)

Smaller sweep than the script: 11 fractions, 5 trials each. The article figure
uses 21 fractions and 30 trials — see `scripts/exp_fraud_demo.py`.

In [ ]:
try:
    fractions = np.linspace(0.0, 1.0, 11)
    curve = detection_power_curve(
        pop,
        fractions=fractions,
        kind="uniform_digit",
        n_trials=5,
        rng=np.random.default_rng(0),
    )

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].plot(curve.fractions, curve.mad, marker="o", color="#3b6ea8")
    for thr, label in [(0.006, "Acceptable"), (0.012, "Marginally"), (0.015, "Non-conformity")]:
        axes[0].axhline(thr, color="grey", linestyle=":", alpha=0.6)
    axes[0].set_xlabel("Contamination fraction")
    axes[0].set_ylabel("MAD")
    axes[0].set_title("MAD vs contamination")
    axes[0].grid(linestyle=":", alpha=0.4)

    axes[1].plot(curve.fractions, curve.chi_square, marker="o", color="#a83b3b")
    axes[1].axhline(15.51, color="grey", linestyle=":", alpha=0.6, label="$\\chi^2_{8, 0.95}$")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Contamination fraction")
    axes[1].set_ylabel("$\\chi^2$")
    axes[1].set_title("$\\chi^2$ vs contamination")
    axes[1].grid(linestyle=":", alpha=0.4, which="both")
    axes[1].legend(frameon=False, fontsize=9)

    axes[2].plot(curve.fractions, curve.rejection_rate_chi2_05, marker="o", color="#3ea872")
    axes[2].axhline(0.05, color="grey", linestyle=":", alpha=0.6)
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].set_xlabel("Contamination fraction")
    axes[2].set_ylabel("Rejection rate")
    axes[2].set_title("Empirical detection power")
    axes[2].grid(linestyle=":", alpha=0.4)

    fig.suptitle(f"Detection power, $n = {curve.n:,}$ ($n_{{trials}} = {curve.n_trials}$)", fontsize=12, y=1.02)
    fig.tight_layout()
    plt.show()

    smallest_rejected = next(
        (float(f) for f, r in zip(curve.fractions, curve.rejection_rate_chi2_05) if r >= 0.95),
        None,
    )
    print(f"Smallest fraction at which $\\chi^2$ rejects ≥ 95% of trials: {smallest_rejected}")
except NameError:
    print("SKIPPED: data not loaded (see cell 2.")

## Reading the results

- **§1.** All three fabrication kinds violate Benford visibly. Round-number
  fabrication is the most realistic for forensic accounting; uniform-digit is
  the cleanest test-case for the detection power calculation.
- **§2.** A single 30 %-contaminated dataset is enough for $\chi^2$ to reject
  decisively, MAD to land in the *Non-conformity* band, and per-digit $Z$ to
  flag specific cells.
- **§3.** The detection-power curve is the article's payoff plot: with the
  GeoNames cities5000 dataset ($n \approx 68{,}000$) and uniform-digit
  fabrication, $\chi^2$ at $\alpha = 0.05$ rejects ~100 % of contaminated
  datasets at fractions below 5–10 %.

**Phase 6** writes the long-form TIL article tying all five derivations and
experiments together. **Phase 7** is publication.